In [1]:
import numpy as np
import xarray as xr
import pandas as pd
import cftime
import dask
import matplotlib.pyplot as plt
import os
import xesmf as xe
import cesmesptools
import ocetrac as ot
import updated_tracker as ut
import pop_tools
import glob
import tqdm

import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.ticker import LongitudeFormatter,LatitudeFormatter
from cartopy.util import add_cyclic_point
import matplotlib
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle
import matplotlib.dates as mdates

from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from matplotlib.animation import FFMpegWriter as fw
from matplotlib.animation import PillowWriter as pw

from IPython.utils import io
from os.path import exists

# Open Dask Cluster

In [2]:
import dask_jobqueue
import distributed

# this first part is checking you're in the right environment
if "client" in locals():
    client.close()
    del client
if "cluster" in locals():
    cluster.close()

# this is where we set up the cluster, your own compute system if you will 
cluster = dask_jobqueue.PBSCluster(
    cores=1,  # The number of cores you want
    memory="15GB",  # Amount of memory
    processes=1,  # How many processes
    queue="casper",  # The type of queue to utilize (/glade/u/apps/dav/opt/usr/bin/execcasper)
    # log_directory="/glade/scratch/dcherian/dask/",  # Use your local directory
    resource_spec="select=1:ncpus=1:mem=15GB",  # Specify resources
    account="uwis0040",  # Input your project ID here / THIS WILL BE DIFFERENT FOR YOU 
    walltime="00:20:00",  # Amount of wall time
    interface="ext",  # Interface to use
)

# this is where we say that we want several of these compute systems,
# because we will have to deal with lots of data and can't just rely on one
cluster.adapt(maximum_jobs=24, minimum_jobs=1) # If you want to force everything to be quicker, 
# increase the number of minimum jobs, but sometimes then it will take a while until you get them assigned 
# (they have to queue), so it's a trade-off
client = distributed.Client(cluster)

# show the client that you have been assigned, you can click on the link and it will show you 
# a dashboard with all the tasks that have to be performed to do your calculation
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/jtcohen/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/jtcohen/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.109:37961,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/jtcohen/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


# Functions

In [3]:
def setup_axes(ax):
    ax.add_feature(cfeature.LAND, facecolor='white', zorder=2)
    ax.coastlines(resolution='110m', color='black', lw=2)
    ax.set_ylabel('latitude')
    ax.set_xlabel('longitude')


def lons_to_360(data, coord='lon'):
    """ Converts longitude coordinates from (-180, 180) to (0, 360)."""
    data.coords[coord] = (360 + (data.coords[coord] % 360)) % 360
    data = data.sortby(data[coord])
    return data


def lons_to_180(data, coord='lon'):
    """ Converts longitude coordinates from (0, 360) to (-180, 180)."""
    data.coords[coord] = (data.coords[coord] + 180) % 360 - 180
    data = data.sortby(data[coord])
    return data


def remove_trend(da, dim, deg=1):
    # detrend along a single dimension
    # return polyfit coefficients and detrended da
    p = da.polyfit(dim=dim, deg=deg, skipna=True)
    coord = da.coords[dim]
    fit = xr.polyval(coord, p.polyfit_coefficients)
    return da - fit


def get_anoms(da):
    clim = da.groupby('time.month').mean('time')
    da_noclim = da.groupby('time.month') - clim
    anoms = remove_trend(da_noclim, dim='time')
    return anoms


def regrid(ds, glat=1, glon=1):
    """
    Inputs:
        ds: xr.DataArray
    Returns:
        Regridded xr.DataArray with coordinates lat and lon
    """
    ds_out = xe.util.grid_global(glon, glat)
    regridder = xe.Regridder(ds, ds_out, 'bilinear', periodic=True)
    regridded = regridder(ds)
    new_coords = regridded.assign_coords({'y': regridded.lat[:, 0].values, 'x': regridded.lon[0].values})
    return new_coords.drop_vars(['lat', 'lon']).rename({'x': 'lon', 'y': 'lat'})

# Load data

In [4]:
firstyear = 1989
lastyear = 2020
field = 'TEMP'

In [5]:
mask = xr.where(~np.isnan(xr.open_dataset('/glade/work/jtcohen/SMYLE_features_premask.r1.TEMP.02.1989-2018.nc')['TEMP'].isel(M=0, L=0, Y=0)), 1, np.nan).drop_vars(['z_t', 'M', 'L', 'Y', 'quantile'])

In [6]:
# def preprocess(ds):
#     return ds['sst'].drop_vars(['zlev'])

In [7]:
# fnames = [np.sort(glob.glob(f'/glade/campaign/collections/rda/data/d277007/avhrr_v2.1/{year}/*')) for year in np.arange(1989, 2021)]

In [8]:
# ds_oisst_daily = xr.concat([xr.open_mfdataset(fn, combine='nested', concat_dim='time', preprocess=preprocess).resample(time='MS').mean() for fn in tqdm.tqdm(fnames)], dim='time')

In [9]:
# oisst = lons_to_180(regrid(ds_oisst_daily)).sortby('lat').squeeze()
# oisst_montime_vals = [cftime.DatetimeNoLeap(t.dt.year, t.dt.month, 15) for t in oisst['time']]
# oisst['time'] = oisst_montime_vals

In [10]:
# oisst.load().to_netcdf('/glade/work/jtcohen/OISST_1deg_monthly_1989-2020.nc')

In [11]:
oisst = xr.open_dataarray('/glade/work/jtcohen/OISST_1deg_monthly_1989-2020.nc')

In [13]:
oisst_30yr = oisst[oisst['time.year'].isin(list(range(1989, 2019)))]
clim = oisst_30yr.groupby('time.month').mean('time')
oisst_30yr_noclim = oisst_30yr.groupby('time.month') - clim
p = oisst_30yr_noclim.polyfit(dim='time', deg=1, skipna=True)

oisst_noclim = oisst.groupby('time.month') - clim
oisst_anom = oisst_noclim - xr.polyval(oisst_noclim.coords['time'], p.polyfit_coefficients)

In [14]:
oisst_anoms = oisst_anom.drop_vars('month').where(mask==1, np.nan)

In [15]:
oisst_anom_1deg = oisst_anoms.where(oisst_anoms!=0, np.nan)
oisst_anom_1deg.loc[dict(lat=slice(-90, -65))] = np.nan
oisst_anom_1deg.loc[dict(lat=slice(65, 90))] = np.nan

In [16]:
mask_1deg = ~np.isnan(oisst_anom_1deg.isel(time=0))

In [17]:
oisst_anom_1deg_30yr = oisst_anom_1deg[oisst_anom_1deg['time.year'].isin(list(range(1989, 2019)))]

# Get threshold

At each point and each lead time, I take the 90th percentile over all the ensembles and initializations (start years).

In [18]:
threshold_val = 0.9

In [19]:
# Define threshold based on 1989-2018 data only
oisst_threshold = oisst_anom_1deg_30yr.groupby('time.month').quantile(threshold_val, dim='time')
oisst_mhw = oisst_anom_1deg.where(oisst_anom_1deg.groupby('time.month')>=oisst_threshold, np.nan)

/glade/work/jtcohen/envs/lib/python3.10/site-packages/numpy/lib/nanfunctions.py:1583: RuntimeWarning: All-NaN slice encountered
  result = np.apply_along_axis(_nanquantile_1d, axis, a, q,


# Applying Ocetrac to observations

## Ocetrac Code

In [20]:
def run_ocetrac(SST_features, mask, radius_val, min_size_quartile_val):
    """
    Run ocetrac on FOSI

    Inputs:
        SST: xr.DataArray with dimensions time, lat, lon
    Outputs:
        blobs: xr.DataArray with dimensions time, lat, lon
    """
    full_masked = SST_features.where(SST_features!=0)
    binary_out_afterlandmask=np.isfinite(full_masked)

    # Run Ocetrac
    xdim = 'lon'
    ydim = 'lat'
    Tracker = ut.Tracker(binary_out_afterlandmask[:,:,:], mask, radius=radius_val, min_size_quartile=min_size_quartile_val, timedim='time', xdim=xdim, ydim=ydim, positive=True)
    blobs = Tracker.track()
    return blobs

# Iterate over radii and save

In [21]:
radius_vals = [1, 2, 3, 4, 5, 6, 7]
min_size_quartile_val = 0

In [22]:
%%time
for radius_val in radius_vals:
    blobs = run_ocetrac(oisst_mhw, mask_1deg, radius_val, min_size_quartile_val)
    outdir = '/glade/work/jtcohen/'
    ds = xr.Dataset({'TEMP': oisst_anom_1deg, 'features': blobs})
    fout = f'OISST_features_premask.r{radius_val}.{field}.{firstyear}-{lastyear}.nc'

    ds.load().to_netcdf(outdir+fout, engine='netcdf4')
    print(f'file saved at {outdir+fout}')
    ds.close()

minimum area: 1.0
inital objects identified 	 113809
final objects tracked 	 34990
file saved at /glade/work/jtcohen/OISST_features_premask.r1.TEMP.1989-2020.nc
minimum area: 1.0
inital objects identified 	 11659
final objects tracked 	 3672
file saved at /glade/work/jtcohen/OISST_features_premask.r2.TEMP.1989-2020.nc
minimum area: 1.0
inital objects identified 	 6524
final objects tracked 	 1673
file saved at /glade/work/jtcohen/OISST_features_premask.r3.TEMP.1989-2020.nc
minimum area: 1.0
inital objects identified 	 4919
final objects tracked 	 1050
file saved at /glade/work/jtcohen/OISST_features_premask.r4.TEMP.1989-2020.nc
minimum area: 1.0
inital objects identified 	 4128
final objects tracked 	 825
file saved at /glade/work/jtcohen/OISST_features_premask.r5.TEMP.1989-2020.nc
minimum area: 1.0
inital objects identified 	 3405
final objects tracked 	 688
file saved at /glade/work/jtcohen/OISST_features_premask.r6.TEMP.1989-2020.nc
minimum area: 1.0
inital objects identified 	 3087